# Hierarchical Cross-Validation Methods## Advanced Validation Techniques for Complex Medical Data StructuresThis notebook covers advanced hierarchical cross-validation methods for medical data with complex nested structures, such as:- **Multi-level clustering** (patients within hospitals within regions)- **Hierarchical time series** (multiple measurements per patient over time)- **Mixed data structures** (spatial + temporal + grouped)- **Custom validation schemes** for specific medical applications### 📚 Learning ObjectivesBy the end of this notebook, you will understand:1. When to use hierarchical CV methods2. How to handle multi-level data structures3. Custom CV implementation for complex medical scenarios4. Performance comparison with standard methods5. Best practices for hierarchical validation

In [ ]:
# Essential importsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom datetime import datetime, timedeltaimport warningswarnings.filterwarnings('ignore')# Scikit-learnfrom sklearn.model_selection import cross_val_score
from trustcv import KFoldMedical, StratifiedKFoldMedical, GroupKFoldMedical, StratifiedKFoldfrom sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifierfrom sklearn.linear_model import LogisticRegressionfrom sklearn.preprocessing import StandardScaler, LabelEncoderfrom sklearn.metrics import (    accuracy_score, precision_score, recall_score, f1_score,    roc_auc_score, average_precision_score, confusion_matrix)# Our medical CV methodsimport syssys.path.append('..')from trustcv.splitters.grouped import GroupKFoldMedical, LeaveOneGroupOutfrom trustcv.splitters.spatial import SpatialBlockCVfrom trustcv.splitters.temporal import TimeSeriesSplit, RollingWindowCV# Set styleplt.style.use('default')sns.set_palette(["#870052", "#FF876F", "#4CAF50", "#2196F3", "#FFA500", "#9C27B0"])# Random seed for reproducibilitynp.random.seed(42)print("🏥 Hierarchical Cross-Validation Methods for Medical Data")print("=" * 60)

## 🏗️ Understanding Hierarchical Data StructuresMedical data often has complex hierarchical structures:```Region├── Hospital A│   ├── Patient 1 (measurements at t1, t2, t3)│   ├── Patient 2 (measurements at t1, t2, t3)│   └── ...├── Hospital B│   ├── Patient N (measurements at t1, t2, t3)│   └── ...└── ...```**Key Challenges:**- Multiple levels of correlation- Different validation requirements at each level- Complex data leakage patterns

In [ ]:
def create_hierarchical_medical_data(n_regions=3, n_hospitals_per_region=4,                                    n_patients_per_hospital=50, n_measurements_per_patient=5):    """    Create realistic hierarchical medical dataset        Structure:    - Regions (geographic areas)    - Hospitals within regions    - Patients within hospitals      - Multiple measurements per patient over time    """    print("🏗️ Creating hierarchical medical dataset...")        all_data = []    patient_counter = 0        # Region characteristics (environmental/socioeconomic factors)    region_effects = {        0: {'name': 'Urban Metro', 'disease_base_rate': 0.15, 'income_level': 0.8},        1: {'name': 'Suburban', 'disease_base_rate': 0.12, 'income_level': 0.6},         2: {'name': 'Rural', 'disease_base_rate': 0.18, 'income_level': 0.4}    }        for region_id in range(n_regions):        region_info = region_effects[region_id]                for hospital_id in range(n_hospitals_per_region):            # Hospital characteristics            hospital_quality = np.random.uniform(0.3, 1.0)  # Quality score            hospital_size = np.random.choice(['Small', 'Medium', 'Large'],                                            p=[0.4, 0.4, 0.2])                        # Hospital effect on outcomes            hospital_effect = hospital_quality * np.random.uniform(0.8, 1.2)                        for patient_idx in range(n_patients_per_hospital):                patient_id = patient_counter                patient_counter += 1                                # Patient demographics (correlated with region)                age = np.random.normal(65, 15)                age = max(18, min(95, age))  # Reasonable bounds                                gender = np.random.choice(['M', 'F'], p=[0.48, 0.52])                                # Socioeconomic factors influenced by region                income_factor = region_info['income_level'] + np.random.normal(0, 0.2)                income_factor = max(0.1, min(1.0, income_factor))                                # Comorbidities (age-dependent)                diabetes = np.random.binomial(1, min(0.6, (age - 30) / 80 + 0.1))                hypertension = np.random.binomial(1, min(0.8, (age - 25) / 70 + 0.15))                heart_disease = np.random.binomial(1, min(0.4, (age - 40) / 60 + 0.05))                                # Generate time series for this patient                measurement_dates = pd.date_range(                    start='2023-01-01',                     periods=n_measurements_per_patient,                     freq='30D'  # Monthly measurements                )                                for measurement_idx, date in enumerate(measurement_dates):                    # Time-varying vitals with trend                    trend_factor = 1 + (measurement_idx * 0.1)  # Health may deteriorate over time                                        # Vital signs (with individual patient baseline)                    patient_baseline = np.random.normal(0, 0.3)                                        systolic_bp = 130 + patient_baseline * 20 + np.random.normal(0, 10) + \                                hypertension * 20 + trend_factor * 5                                        heart_rate = 72 + patient_baseline * 15 + np.random.normal(0, 8) + \                               heart_disease * 12 + trend_factor * 3                                        # Lab values                    glucose = 95 + patient_baseline * 25 + np.random.normal(0, 15) + \                            diabetes * 50 + trend_factor * 10                                        cholesterol = 180 + patient_baseline * 30 + np.random.normal(0, 20) + \                                heart_disease * 40 + age * 0.5                                        # Calculate risk score (complex interaction of factors)                    risk_score = (                        0.02 * age +                         0.3 * (gender == 'M') +                        0.4 * diabetes +                        0.3 * hypertension +                        0.5 * heart_disease +                        0.001 * (systolic_bp - 120) +                        0.002 * (glucose - 100) +                        0.001 * (cholesterol - 200) +                        region_info['disease_base_rate'] +                        (1 - hospital_effect) * 0.2 +  # Poor hospital increases risk                        (1 - income_factor) * 0.15 +   # Low income increases risk                        trend_factor * 0.1 +           # Risk increases over time                        np.random.normal(0, 0.1)       # Random noise                    )                                        # Convert to probability and generate outcome                    prob = 1 / (1 + np.exp(-risk_score))  # Logistic function                    adverse_outcome = np.random.binomial(1, prob)                                        all_data.append({                        # Hierarchical identifiers                        'region_id': region_id,                        'region_name': region_info['name'],                        'hospital_id': f"{region_id}_{hospital_id}",                        'patient_id': patient_id,                        'measurement_date': date,                        'measurement_idx': measurement_idx,                                                # Demographics                        'age': age,                        'gender': gender,                        'income_factor': income_factor,                                                # Hospital characteristics                        'hospital_quality': hospital_quality,                        'hospital_size': hospital_size,                                                # Comorbidities                        'diabetes': diabetes,                        'hypertension': hypertension,                        'heart_disease': heart_disease,                                                # Vital signs and labs                        'systolic_bp': systolic_bp,                        'heart_rate': heart_rate,                        'glucose': glucose,                        'cholesterol': cholesterol,                                                # Derived features                        'risk_score': risk_score,                                                # Target variable                        'adverse_outcome': adverse_outcome                    })        df = pd.DataFrame(all_data)        print(f"📊 Generated hierarchical dataset:")    print(f"   Shape: {df.shape}")    print(f"   Regions: {df['region_id'].nunique()}")    print(f"   Hospitals: {df['hospital_id'].nunique()}")    print(f"   Patients: {df['patient_id'].nunique()}")    print(f"   Total measurements: {len(df)}")    print(f"   Adverse outcome rate: {df['adverse_outcome'].mean():.2%}")        return df# Generate hierarchical datasetdf = create_hierarchical_medical_data(    n_regions=3,     n_hospitals_per_region=4,     n_patients_per_hospital=50,     n_measurements_per_patient=5)

## 📊 Exploratory Analysis of Hierarchical Structure

In [ ]:
def analyze_hierarchical_structure(df):    """Analyze the hierarchical structure and correlations"""        fig, axes = plt.subplots(2, 2, figsize=(15, 12))        # Plot 1: Outcome rates by region    ax1 = axes[0, 0]    region_outcomes = df.groupby('region_name')['adverse_outcome'].agg(['mean', 'count']).reset_index()    bars1 = ax1.bar(region_outcomes['region_name'], region_outcomes['mean'],                     color=['#870052', '#FF876F', '#4CAF50'])    ax1.set_ylabel('Adverse Outcome Rate')    ax1.set_title('Outcome Rates by Region')    ax1.set_ylim(0, region_outcomes['mean'].max() * 1.2)        # Add sample size annotations    for bar, count in zip(bars1, region_outcomes['count']):        height = bar.get_height()        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.005,                f'n={count}', ha='center', va='bottom', fontsize=10)        # Plot 2: Outcome rates by hospital quality    ax2 = axes[0, 1]    df['quality_quartile'] = pd.qcut(df['hospital_quality'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])    quality_outcomes = df.groupby('quality_quartile')['adverse_outcome'].mean()    bars2 = ax2.bar(quality_outcomes.index, quality_outcomes.values,                     color=['#FF4444', '#FF876F', '#4CAF50', '#2196F3'])    ax2.set_ylabel('Adverse Outcome Rate')    ax2.set_title('Outcome Rates by Hospital Quality Quartile')    ax2.set_xlabel('Hospital Quality Quartile (Q1=Lowest, Q4=Highest)')        # Plot 3: Patient-level variability    ax3 = axes[1, 0]    patient_outcomes = df.groupby('patient_id')['adverse_outcome'].mean()    ax3.hist(patient_outcomes, bins=20, alpha=0.7, color='#870052', edgecolor='black')    ax3.set_xlabel('Patient-Level Outcome Rate')    ax3.set_ylabel('Number of Patients')    ax3.set_title('Distribution of Patient-Level Outcome Rates')    ax3.axvline(patient_outcomes.mean(), color='red', linestyle='--',                 label=f'Mean: {patient_outcomes.mean():.3f}')    ax3.legend()        # Plot 4: Temporal trends    ax4 = axes[1, 1]    temporal_outcomes = df.groupby('measurement_idx')['adverse_outcome'].mean()    ax4.plot(temporal_outcomes.index, temporal_outcomes.values,              marker='o', linewidth=2, markersize=8, color='#870052')    ax4.set_xlabel('Measurement Time Point')    ax4.set_ylabel('Adverse Outcome Rate')    ax4.set_title('Temporal Trend in Outcome Rates')    ax4.grid(True, alpha=0.3)        # Add trend line    z = np.polyfit(temporal_outcomes.index, temporal_outcomes.values, 1)    p = np.poly1d(z)    ax4.plot(temporal_outcomes.index, p(temporal_outcomes.index), "--",              color='red', alpha=0.8, label=f'Trend: {z[0]:+.4f} per time point')    ax4.legend()        plt.tight_layout()    plt.show()        # Statistical analysis    print("\n📈 Hierarchical Structure Analysis:")    print("=" * 40)        # Region-level variance    region_variance = region_outcomes['mean'].var()    print(f"Region-level outcome variance: {region_variance:.6f}")        # Hospital-level variance (within regions)    hospital_outcomes = df.groupby('hospital_id')['adverse_outcome'].mean()    hospital_variance = hospital_outcomes.var()    print(f"Hospital-level outcome variance: {hospital_variance:.6f}")        # Patient-level variance (within hospitals)    patient_variance = patient_outcomes.var()    print(f"Patient-level outcome variance: {patient_variance:.6f}")        # Intraclass correlation coefficients    total_variance = df['adverse_outcome'].var()    region_icc = region_variance / total_variance    hospital_icc = hospital_variance / total_variance    patient_icc = patient_variance / total_variance        print(f"\n🔗 Intraclass Correlations (ICC):")    print(f"   Region ICC: {region_icc:.4f}")    print(f"   Hospital ICC: {hospital_icc:.4f}")    print(f"   Patient ICC: {patient_icc:.4f}")        if region_icc > 0.05:        print("   ⚠️ Significant regional clustering - use region-aware CV")    if hospital_icc > 0.05:        print("   ⚠️ Significant hospital clustering - use hospital-aware CV")    if patient_icc > 0.10:        print("   ⚠️ Significant patient clustering - use patient-aware CV")analyze_hierarchical_structure(df)

## 🔀 Hierarchical Cross-Validation Strategies### Strategy 1: Single-Level Validation (WRONG!)Ignores hierarchical structure - leads to data leakage

In [ ]:
# Prepare featuresfeature_cols = [    'age', 'diabetes', 'hypertension', 'heart_disease',    'systolic_bp', 'heart_rate', 'glucose', 'cholesterol',    'hospital_quality', 'income_factor', 'measurement_idx']# Encode categorical variablesdf_model = df.copy()le_gender = LabelEncoder()df_model['gender_encoded'] = le_gender.fit_transform(df_model['gender'])feature_cols.append('gender_encoded')le_size = LabelEncoder()df_model['hospital_size_encoded'] = le_size.fit_transform(df_model['hospital_size'])feature_cols.append('hospital_size_encoded')X = df_model[feature_cols].valuesy = df_model['adverse_outcome'].values# Standardize featuresscaler = StandardScaler()X_scaled = scaler.fit_transform(X)print(f"📊 Model Dataset:")print(f"   Features: {X_scaled.shape[1]}")print(f"   Samples: {X_scaled.shape[0]}")print(f"   Positive rate: {y.mean():.3f}")

In [ ]:
def evaluate_single_level_cv(X, y):    """Standard stratified K-fold (WRONG for hierarchical data)"""        print("\n❌ Strategy 1: Standard Stratified K-Fold (WRONG!)")    print("-" * 50)        cv = StratifiedKFoldMedical(n_splits=5, shuffle=True, random_state=42)        models = {        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),        'Gradient Boosting': GradientBoostingClassifier(random_state=42),        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)    }        results = {}        for name, model in models.items():        scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')        results[name] = scores        print(f"{name:20s}: AUC = {scores.mean():.4f} ± {scores.std():.4f}")        print("\n⚠️ These results are OPTIMISTICALLY BIASED!")    print("   Same patients appear in both train and test sets")    print("   Same hospitals appear in both train and test sets")    print("   Model learns hospital and patient-specific patterns")        return resultssingle_level_results = evaluate_single_level_cv(X_scaled, y)

### Strategy 2: Patient-Level Validation

In [ ]:
def evaluate_patient_level_cv(X, y, patient_ids):    """Patient-level grouped cross-validation"""        print("\n✅ Strategy 2: Patient-Level Cross-Validation")    print("-" * 50)    print("No patient appears in both train and test sets")        cv = GroupKFoldMedical(n_splits=5)    results = {}        models = {        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),        'Gradient Boosting': GradientBoostingClassifier(random_state=42),        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)    }        for name, model in models.items():        fold_scores = []                for train_idx, test_idx in cv.split(X, y, groups=patient_ids):            model.fit(X[train_idx], y[train_idx])            y_pred_proba = model.predict_proba(X[test_idx])[:, 1]            score = roc_auc_score(y[test_idx], y_pred_proba)            fold_scores.append(score)                results[name] = np.array(fold_scores)        print(f"{name:20s}: AUC = {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")                # Check for patient leakage        sample_train_idx, sample_test_idx = next(cv.split(X, y, groups=patient_ids))        train_patients = set(patient_ids[sample_train_idx])        test_patients = set(patient_ids[sample_test_idx])        assert len(train_patients & test_patients) == 0, "Patient leakage detected!"        print("\n✅ No patient leakage - realistic within-hospital generalization")    return resultspatient_level_results = evaluate_patient_level_cv(X_scaled, y, df_model['patient_id'].values)

### Strategy 3: Hospital-Level Validation

In [ ]:
def evaluate_hospital_level_cv(X, y, hospital_ids):    """Hospital-level grouped cross-validation"""        print("\n✅ Strategy 3: Hospital-Level Cross-Validation")    print("-" * 50)    print("No hospital appears in both train and test sets")        cv = GroupKFoldMedical(n_splits=5)  # Will use available hospitals    results = {}        models = {        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),        'Gradient Boosting': GradientBoostingClassifier(random_state=42),        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)    }        for name, model in models.items():        fold_scores = []                for train_idx, test_idx in cv.split(X, y, groups=hospital_ids):            model.fit(X[train_idx], y[train_idx])            y_pred_proba = model.predict_proba(X[test_idx])[:, 1]            score = roc_auc_score(y[test_idx], y_pred_proba)            fold_scores.append(score)                results[name] = np.array(fold_scores)        print(f"{name:20s}: AUC = {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")                # Check for hospital leakage        sample_train_idx, sample_test_idx = next(cv.split(X, y, groups=hospital_ids))        train_hospitals = set(hospital_ids[sample_train_idx])        test_hospitals = set(hospital_ids[sample_test_idx])        assert len(train_hospitals & test_hospitals) == 0, "Hospital leakage detected!"        print("\n✅ No hospital leakage - realistic cross-hospital generalization")    return resultshospital_level_results = evaluate_hospital_level_cv(X_scaled, y, df_model['hospital_id'].values)

### Strategy 4: Region-Level Validation (Most Conservative)

In [ ]:
def evaluate_region_level_cv(X, y, region_ids):    """Region-level validation - most conservative"""        print("\n✅ Strategy 4: Region-Level Cross-Validation (Leave-One-Region-Out)")    print("-" * 50)    print("Each region is held out entirely for testing")        cv = LeaveOneGroupOut()    results = {}        models = {        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),        'Gradient Boosting': GradientBoostingClassifier(random_state=42),        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)    }        region_names = ['Urban Metro', 'Suburban', 'Rural']        for name, model in models.items():        fold_scores = []        fold_details = []                for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y, groups=region_ids)):            model.fit(X[train_idx], y[train_idx])            y_pred_proba = model.predict_proba(X[test_idx])[:, 1]            score = roc_auc_score(y[test_idx], y_pred_proba)            fold_scores.append(score)                        test_region = region_ids[test_idx[0]]            fold_details.append({                'region': region_names[test_region],                'score': score,                'test_size': len(test_idx)            })                results[name] = {            'scores': np.array(fold_scores),            'details': fold_details        }                print(f"\n{name}:")        for detail in fold_details:            print(f"  {detail['region']:12s}: AUC = {detail['score']:.4f} (n={detail['test_size']})")        print(f"  {'Mean':12s}: AUC = {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")        print("\n✅ Most conservative - tests geographic generalization")    return resultsregion_level_results = evaluate_region_level_cv(X_scaled, y, df_model['region_id'].values)

### Strategy 5: Mixed Hierarchical Validation

In [ ]:
class HierarchicalCV:    """    Custom hierarchical cross-validation that combines multiple levels        This implementation:    1. First splits by regions (ensuring geographic generalization)    2. Within each region, splits by hospitals    3. Ensures no patient appears in both train and test    """        def __init__(self, n_splits=3, test_region_size=0.33):        self.n_splits = n_splits        self.test_region_size = test_region_size        def split(self, X, y=None, groups=None, region_ids=None, hospital_ids=None, patient_ids=None):        """        Generate hierarchical train/test splits                Parameters:        -----------        X : array-like            Features        y : array-like            Target variable        groups : array-like            Primary grouping variable (e.g., patient_ids)        region_ids : array-like            Region identifiers        hospital_ids : array-like            Hospital identifiers        patient_ids : array-like            Patient identifiers        """                n_samples = len(X)        unique_regions = np.unique(region_ids)                for fold in range(self.n_splits):            # Select test region(s) - round robin            test_region = unique_regions[fold % len(unique_regions)]            train_regions = [r for r in unique_regions if r != test_region]                        # Get all samples from test region            test_region_mask = region_ids == test_region            test_candidates = np.where(test_region_mask)[0]                        # Get unique hospitals in test region            test_region_hospitals = np.unique(hospital_ids[test_region_mask])                        # Select subset of hospitals for testing (to have some for training)            n_test_hospitals = max(1, int(len(test_region_hospitals) * 0.5))            np.random.seed(42 + fold)  # Reproducible selection            test_hospitals = np.random.choice(test_region_hospitals, n_test_hospitals, replace=False)                        # Final test set: samples from selected hospitals in test region            test_mask = test_region_mask & np.isin(hospital_ids, test_hospitals)            test_idx = np.where(test_mask)[0]                        # Training set: all other samples            train_idx = np.where(~test_mask)[0]                        # Ensure no patient leakage            test_patients = set(patient_ids[test_idx])            train_mask_clean = ~np.isin(patient_ids, list(test_patients))            train_idx = np.where(train_mask_clean)[0]                        # Ensure no hospital leakage in test hospitals            train_mask_hospital = ~np.isin(hospital_ids, test_hospitals)            train_idx = np.intersect1d(train_idx, np.where(train_mask_hospital)[0])                        yield train_idx, test_idx        def get_n_splits(self, X=None, y=None, groups=None):        return self.n_splitsdef evaluate_hierarchical_cv(X, y, region_ids, hospital_ids, patient_ids):    """Mixed hierarchical cross-validation"""        print("\n🏗️ Strategy 5: Mixed Hierarchical Cross-Validation")    print("-" * 50)    print("Combines region + hospital + patient level constraints")        cv = HierarchicalCV(n_splits=3)  # Limited by number of regions    results = {}        models = {        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),        'Gradient Boosting': GradientBoostingClassifier(random_state=42),        'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)    }        region_names = ['Urban Metro', 'Suburban', 'Rural']        for name, model in models.items():        fold_scores = []        fold_details = []                for fold_idx, (train_idx, test_idx) in enumerate(cv.split(            X, y,             groups=patient_ids,            region_ids=region_ids,             hospital_ids=hospital_ids,             patient_ids=patient_ids        )):            model.fit(X[train_idx], y[train_idx])            y_pred_proba = model.predict_proba(X[test_idx])[:, 1]            score = roc_auc_score(y[test_idx], y_pred_proba)            fold_scores.append(score)                        # Analysis of split            test_regions = set(region_ids[test_idx])            test_hospitals = set(hospital_ids[test_idx])            test_patients = set(patient_ids[test_idx])                        train_regions = set(region_ids[train_idx])            train_hospitals = set(hospital_ids[train_idx])            train_patients = set(patient_ids[train_idx])                        fold_details.append({                'score': score,                'test_size': len(test_idx),                'train_size': len(train_idx),                'test_regions': len(test_regions),                'test_hospitals': len(test_hospitals),                'test_patients': len(test_patients),                'region_overlap': len(test_regions & train_regions),                'hospital_overlap': len(test_hospitals & train_hospitals),                'patient_overlap': len(test_patients & train_patients)            })                results[name] = {            'scores': np.array(fold_scores),            'details': fold_details        }                print(f"\n{name}:")        for i, detail in enumerate(fold_details):            print(f"  Fold {i+1}: AUC = {detail['score']:.4f} "                  f"(train: {detail['train_size']}, test: {detail['test_size']})")            print(f"    Overlaps - Regions: {detail['region_overlap']}, "                  f"Hospitals: {detail['hospital_overlap']}, "                  f"Patients: {detail['patient_overlap']}")                print(f"  {'Mean':12s}: AUC = {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")        print("\n✅ Most realistic - tests multi-level generalization")    return resultshierarchical_results = evaluate_hierarchical_cv(    X_scaled, y,     df_model['region_id'].values,     df_model['hospital_id'].values,     df_model['patient_id'].values)

## 📊 Comprehensive Results Comparison

In [ ]:
def visualize_hierarchical_results():    """Comprehensive visualization of all CV strategies"""        fig, axes = plt.subplots(2, 2, figsize=(15, 12))        # Collect all results for Random Forest    rf_results = {        'Standard K-Fold\n(BIASED)': single_level_results['Random Forest'],        'Patient-Level': patient_level_results['Random Forest'],        'Hospital-Level': hospital_level_results['Random Forest'],        'Region-Level': region_level_results['Random Forest']['scores'],        'Hierarchical': hierarchical_results['Random Forest']['scores']    }        # Plot 1: AUC comparison    ax1 = axes[0, 0]    methods = list(rf_results.keys())    scores_data = [rf_results[method] for method in methods]        bp1 = ax1.boxplot(scores_data, labels=[m.replace('\n', ' ') for m in methods],                       patch_artist=True)        colors = ['#FF4444', '#870052', '#FF876F', '#4CAF50', '#2196F3']    for patch, color in zip(bp1['boxes'], colors):        patch.set_facecolor(color)        patch.set_alpha(0.7)        ax1.set_ylabel('AUC Score')    ax1.set_title('Random Forest: CV Strategy Comparison')    ax1.tick_params(axis='x', rotation=45)    ax1.grid(True, alpha=0.3)        # Add mean values as text    for i, (method, scores) in enumerate(rf_results.items()):        mean_score = np.mean(scores)        ax1.text(i+1, mean_score + 0.01, f'{mean_score:.3f}',                 ha='center', va='bottom', fontweight='bold')        # Plot 2: Bias estimation (difference from most conservative)    ax2 = axes[0, 1]    conservative_score = np.mean(rf_results['Hierarchical'])    biases = [(np.mean(scores) - conservative_score) for scores in scores_data]        bars = ax2.bar(range(len(methods)), biases,                    color=['#FF4444' if b > 0.02 else '#4CAF50' for b in biases])    ax2.set_xticks(range(len(methods)))    ax2.set_xticklabels([m.replace('\n', ' ') for m in methods], rotation=45)    ax2.set_ylabel('Optimistic Bias (vs Hierarchical)')    ax2.set_title('Estimation Bias by CV Strategy')    ax2.axhline(y=0, color='black', linestyle='--', alpha=0.5)    ax2.grid(True, alpha=0.3)        # Add value labels    for bar, bias in zip(bars, biases):        height = bar.get_height()        ax2.text(bar.get_x() + bar.get_width()/2.,                 height + (0.002 if height >= 0 else -0.005),                f'{bias:+.3f}', ha='center',                 va='bottom' if height >= 0 else 'top')        # Plot 3: Variance comparison    ax3 = axes[1, 0]    variances = [np.std(scores) for scores in scores_data]    bars3 = ax3.bar(range(len(methods)), variances, color=colors, alpha=0.7)    ax3.set_xticks(range(len(methods)))    ax3.set_xticklabels([m.replace('\n', ' ') for m in methods], rotation=45)    ax3.set_ylabel('Standard Deviation')    ax3.set_title('Variance by CV Strategy')    ax3.grid(True, alpha=0.3)        # Plot 4: Sample size impact    ax4 = axes[1, 1]        # Simulate different sample sizes    sample_sizes = [100, 200, 500, 1000, 2000]        # Estimate bias reduction with sample size (simplified model)    standard_bias = [0.15, 0.12, 0.08, 0.05, 0.03]    patient_bias = [0.08, 0.06, 0.04, 0.02, 0.01]    hospital_bias = [0.05, 0.04, 0.03, 0.02, 0.015]        ax4.plot(sample_sizes, standard_bias, 'o-', label='Standard K-Fold',              color='#FF4444', linewidth=2, markersize=6)    ax4.plot(sample_sizes, patient_bias, 's-', label='Patient-Level',              color='#870052', linewidth=2, markersize=6)    ax4.plot(sample_sizes, hospital_bias, '^-', label='Hospital-Level',              color='#FF876F', linewidth=2, markersize=6)        ax4.set_xlabel('Sample Size')    ax4.set_ylabel('Estimated Bias')    ax4.set_title('CV Bias vs Sample Size')    ax4.set_xscale('log')    ax4.legend()    ax4.grid(True, alpha=0.3)        plt.tight_layout()    plt.show()        # Statistical summary    print("\n📊 Hierarchical CV Results Summary:")    print("=" * 50)        for method, scores in rf_results.items():        mean_score = np.mean(scores)        std_score = np.std(scores)        bias = mean_score - conservative_score                status = "⚠️ BIASED" if bias > 0.02 else "✅ OK"        print(f"{method:20s}: {mean_score:.4f} ± {std_score:.4f} (bias: {bias:+.4f}) {status}")visualize_hierarchical_results()

## 🎯 Practical Guidelines for Hierarchical CV### Decision Framework

In [ ]:
def hierarchical_cv_decision_guide():    """    Interactive decision guide for choosing hierarchical CV strategy    """        print("🎯 HIERARCHICAL CROSS-VALIDATION DECISION GUIDE")    print("=" * 55)        print("\n❓ Choose your CV strategy based on deployment scenario:\n")        scenarios = {        "Within-Hospital Deployment": {            "description": "Model will be used in the same hospitals where it was trained",            "recommendation": "Patient-Level CV",            "rationale": "Ensures no patient-level overfitting, realistic within-hospital performance",            "example": "ICU deterioration prediction for current patients"        },                "Cross-Hospital Deployment": {            "description": "Model will be deployed to hospitals not in training data",            "recommendation": "Hospital-Level CV",            "rationale": "Tests generalization across different hospital systems and practices",            "example": "Multi-site clinical decision support system"        },                "Geographic Expansion": {            "description": "Model will be deployed in different regions/countries",            "recommendation": "Region-Level CV",            "rationale": "Accounts for population differences, healthcare systems, environmental factors",            "example": "Pandemic spread prediction across regions"        },                "Research Publication": {            "description": "Results will be published for scientific community",            "recommendation": "Hierarchical CV (Multi-Level)",            "rationale": "Most conservative, prevents multiple types of data leakage",            "example": "Novel biomarker discovery study"        }    }        for scenario, details in scenarios.items():        print(f"📋 {scenario}")        print(f"   Description: {details['description']}")        print(f"   ✅ Recommended: {details['recommendation']}")        print(f"   💡 Rationale: {details['rationale']}")        print(f"   🏥 Example: {details['example']}")        print()        print("⚠️  NEVER USE Standard K-Fold for hierarchical medical data!")    print("   It will give optimistically biased results due to data leakage.\n")        # Implementation checklist    print("✅ IMPLEMENTATION CHECKLIST:")    print("-" * 25)        checklist = [        "Identify all hierarchical levels in your data",        "Determine the appropriate validation level for your use case",        "Verify no data leakage between train/test at chosen level",        "Check for sufficient samples at each hierarchical level",        "Consider class balance within each fold",        "Document your CV strategy in methods section",        "Report confidence intervals, not just point estimates",        "Compare with at least one other CV strategy as sensitivity analysis"    ]        for i, item in enumerate(checklist, 1):        print(f"   {i}. {item}")hierarchical_cv_decision_guide()

## 🔧 Custom Implementation Template

In [ ]:
class CustomHierarchicalCV:    """    Template for custom hierarchical cross-validation        Modify this template for your specific hierarchical structure    """        def __init__(self, hierarchy_levels, n_splits=5, test_size=0.2, random_state=None):        """        Parameters:        -----------        hierarchy_levels : list of str            Names of hierarchical levels from highest to lowest            e.g., ['region', 'hospital', 'patient']        n_splits : int            Number of CV folds        test_size : float            Proportion of data for testing        random_state : int            Random seed for reproducibility        """        self.hierarchy_levels = hierarchy_levels        self.n_splits = n_splits        self.test_size = test_size        self.random_state = random_state        def split(self, X, y=None, **hierarchy_groups):        """        Generate hierarchical train/test splits                Parameters:        -----------        X : array-like            Feature matrix        y : array-like, optional            Target variable        **hierarchy_groups : dict            Hierarchy group arrays, e.g., region_ids=array, hospital_ids=array                Yields:        -------        train_idx, test_idx : arrays            Training and testing indices for each fold        """                if self.random_state is not None:            np.random.seed(self.random_state)                n_samples = len(X)                # Determine primary grouping level (highest in hierarchy)        primary_level = self.hierarchy_levels[0]        primary_groups = hierarchy_groups.get(f'{primary_level}_ids')                if primary_groups is None:            raise ValueError(f"Must provide {primary_level}_ids for hierarchical CV")                unique_primary_groups = np.unique(primary_groups)        n_primary_groups = len(unique_primary_groups)                # Create folds at primary level        fold_size = max(1, int(n_primary_groups * self.test_size))                for fold in range(self.n_splits):            # Select test groups for this fold            if fold < n_primary_groups:                # Round-robin selection                test_groups = [unique_primary_groups[fold % n_primary_groups]]            else:                # Random selection if more folds than groups                test_groups = np.random.choice(unique_primary_groups, fold_size, replace=False)                        # Get indices for test groups            test_mask = np.isin(primary_groups, test_groups)                        # Apply additional hierarchy constraints            train_mask = ~test_mask                        # Ensure no leakage at lower hierarchical levels            for level in self.hierarchy_levels[1:]:                level_groups = hierarchy_groups.get(f'{level}_ids')                if level_groups is not None:                    test_level_groups = set(level_groups[test_mask])                    train_mask = train_mask & ~np.isin(level_groups, list(test_level_groups))                        train_idx = np.where(train_mask)[0]            test_idx = np.where(test_mask)[0]                        if len(train_idx) == 0 or len(test_idx) == 0:                continue  # Skip invalid folds                        yield train_idx, test_idx        def validate_split(self, train_idx, test_idx, **hierarchy_groups):        """        Validate that there's no data leakage in the split                Returns:        --------        dict : Validation results        """        results = {}                for level in self.hierarchy_levels:            level_groups = hierarchy_groups.get(f'{level}_ids')            if level_groups is not None:                train_groups = set(level_groups[train_idx])                test_groups = set(level_groups[test_idx])                overlap = train_groups & test_groups                                results[level] = {                    'train_groups': len(train_groups),                    'test_groups': len(test_groups),                    'overlap': len(overlap),                    'leakage_free': len(overlap) == 0                }                return results# Example usageprint("🔧 Custom Hierarchical CV Template")print("=" * 35)# Initialize custom CVcustom_cv = CustomHierarchicalCV(    hierarchy_levels=['region', 'hospital', 'patient'],    n_splits=3,    test_size=0.33,    random_state=42)# Demonstrate validationprint("\n📋 Split Validation Example:")for fold, (train_idx, test_idx) in enumerate(custom_cv.split(    X_scaled,    y,    region_ids=df_model['region_id'].values,    hospital_ids=df_model['hospital_id'].values,    patient_ids=df_model['patient_id'].values)):    print(f"\nFold {fold + 1}:")        validation = custom_cv.validate_split(        train_idx, test_idx,        region_ids=df_model['region_id'].values,        hospital_ids=df_model['hospital_id'].values,        patient_ids=df_model['patient_id'].values    )        for level, stats in validation.items():        status = "✅" if stats['leakage_free'] else "❌"        print(f"  {level:8s}: {stats['train_groups']:3d} train, {stats['test_groups']:3d} test, "              f"overlap: {stats['overlap']:2d} {status}")        print(f"  Samples: {len(train_idx):4d} train, {len(test_idx):4d} test")        if fold >= 2:  # Limit output        break

## 💡 Key Takeaways and Best Practices### Summary of Hierarchical CV Strategies

In [ ]:
def print_hierarchical_cv_summary():    """    Print comprehensive summary of hierarchical CV methods    """        print("🎯 HIERARCHICAL CROSS-VALIDATION SUMMARY")    print("=" * 45)        print("\n📊 PERFORMANCE HIERARCHY (Typical Order):")    print("   1. Standard K-Fold (BIASED) - Highest scores, unrealistic")    print("   2. Patient-Level CV - Good for within-hospital deployment")    print("   3. Hospital-Level CV - Realistic for cross-hospital deployment")    print("   4. Region-Level CV - Conservative for geographic expansion")    print("   5. Hierarchical CV - Most realistic for complex deployment")        print("\n🎯 WHEN TO USE EACH METHOD:")    print("-" * 30)        use_cases = {        "Patient-Level CV": [            "Clinical decision support within same hospitals",            "Longitudinal patient monitoring",            "Personalized medicine applications",            "Quality improvement initiatives"        ],                "Hospital-Level CV": [            "Multi-site clinical trials",            "Healthcare network implementations",            "Regulatory submissions",            "Vendor product validation"        ],                "Region-Level CV": [            "International deployment",            "Public health surveillance",            "Pandemic response systems",            "Population health studies"        ],                "Hierarchical CV": [            "Research publications",            "FDA/EMA submissions",            "Novel method validation",            "Conservative risk assessment"        ]    }        for method, cases in use_cases.items():        print(f"\n📋 {method}:")        for case in cases:            print(f"   • {case}")        print("\n⚠️  COMMON MISTAKES TO AVOID:")    print("-" * 35)        mistakes = [        "Using standard K-fold on hierarchical data",        "Ignoring temporal order in longitudinal data",        "Not checking for data leakage at all levels",        "Choosing CV strategy after seeing results",        "Not reporting confidence intervals",        "Mixing train/test patients from same family/group",        "Using different CV strategies for hyperparameters vs final evaluation",        "Not validating splits for leakage"    ]        for i, mistake in enumerate(mistakes, 1):        print(f"   {i}. ❌ {mistake}")        print("\n✅ BEST PRACTICES:")    print("-" * 20)        best_practices = [        "Pre-register your CV strategy before analysis",        "Always validate splits for data leakage",        "Report results from multiple CV strategies",        "Use conservative methods for high-stakes applications",        "Document all hierarchical levels in your data",        "Check class balance within each fold",        "Save and version your CV splits",        "Include CV strategy in reproducibility documentation"    ]        for i, practice in enumerate(best_practices, 1):        print(f"   {i}. ✅ {practice}")        print("\n🏥 MEDICAL ML DEPLOYMENT CHECKLIST:")    print("-" * 40)        deployment_checklist = [        "Identify your deployment context (within-hospital, cross-hospital, etc.)",        "Choose appropriate CV strategy based on deployment context",        "Validate that CV strategy matches real-world constraints",        "Test model performance degrades gracefully across hierarchy levels",        "Document potential biases and limitations",        "Plan for model updates when new sites/regions are added",        "Establish monitoring for distribution shift",        "Create retraining protocols for different hierarchical levels"    ]        for i, item in enumerate(deployment_checklist, 1):        print(f"   {i}. {item}")        print("\n🎖️ Remember: The goal is not to get the highest CV score,")    print("   but to get the most realistic estimate for your deployment scenario!")print_hierarchical_cv_summary()

## 🔬 Research ExtensionsFuture directions for hierarchical cross-validation research:

In [ ]:
def hierarchical_cv_research_directions():    """    Outline research directions for hierarchical CV    """        print("🔬 RESEARCH DIRECTIONS FOR HIERARCHICAL CV")    print("=" * 45)        directions = {        "Adaptive Hierarchical CV": {            "description": "CV strategies that adapt based on data structure",            "challenges": ["Automatic hierarchy detection", "Dynamic fold allocation", "Bias-variance trade-offs"],            "applications": ["Federated learning", "Multi-modal data", "Dynamic healthcare networks"]        },                "Bayesian Hierarchical CV": {            "description": "Incorporate uncertainty at multiple hierarchical levels",            "challenges": ["Prior specification", "Computational complexity", "Interpretation"],            "applications": ["Small sample studies", "Rare diseases", "Personalized medicine"]        },                "Temporal-Spatial-Hierarchical CV": {            "description": "Combine temporal, spatial, and group constraints simultaneously",            "challenges": ["Multi-constraint optimization", "Sufficient sample sizes", "Validation complexity"],            "applications": ["Pandemic modeling", "Environmental health", "Precision agriculture"]        },                "Fairness-Aware Hierarchical CV": {            "description": "Ensure equitable performance across hierarchical groups",            "challenges": ["Defining fairness metrics", "Multiple protected attributes", "Intersectionality"],            "applications": ["Health equity", "Algorithmic bias detection", "Regulatory compliance"]        },                "Federated Hierarchical CV": {            "description": "CV for models trained across institutions without data sharing",            "challenges": ["Privacy preservation", "Communication efficiency", "Heterogeneous data"],            "applications": ["Multi-site trials", "International collaborations", "Rare disease consortiums"]        }    }        for direction, details in directions.items():        print(f"\n🎯 {direction}")        print(f"   Description: {details['description']}")        print(f"   Key Challenges:")        for challenge in details['challenges']:            print(f"     • {challenge}")        print(f"   Applications:")        for app in details['applications']:            print(f"     • {app}")        print("\n📝 OPEN QUESTIONS:")    print("-" * 20)        questions = [        "How to optimally allocate samples across hierarchical levels?",        "What is the theoretical bias-variance trade-off for different strategies?",        "How to validate hierarchical CV methods themselves?",        "Can we develop adaptive methods that optimize hierarchy constraints?",        "How to extend to continuous hierarchical structures?",        "What are the computational limits for large-scale hierarchical CV?",        "How to integrate domain expertise into hierarchy specification?",        "Can we automate CV strategy selection based on data characteristics?"    ]        for i, question in enumerate(questions, 1):        print(f"   {i}. {question}")        print("\n🏆 CALL TO ACTION:")    print("The ML community needs more rigorous validation methods.")    print("Hierarchical CV is a crucial step toward more reliable and")    print("generalizable AI systems. Join us in advancing this field!")hierarchical_cv_research_directions()

## 🎓 ConclusionHierarchical cross-validation is essential for realistic evaluation of ML models with complex data structures. Key points:1. **Choose CV strategy based on deployment context** - not just highest performance2. **Always validate for data leakage** at all hierarchical levels3. **Report results from multiple strategies** for transparency4. **Use conservative methods** for high-stakes medical applications5. **Document your rationale** for CV strategy selectionRemember: **The goal is realistic performance estimation, not optimistic bias!**---### 📚 Further Reading- Roberts et al. (2017). "Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure"- Varoquaux et al. (2017). "Assessing and tuning brain decoders: Cross-validation, caveats, and guidelines"- Bates et al. (2023). "Cross-validation for assessing the transferability of species distribution models"- trustcv documentation: Advanced validation strategies### 🔗 Next Steps1. Try hierarchical CV on your own medical datasets2. Implement custom CV strategies for your specific hierarchy3. Compare results across different validation levels4. Document and share your CV strategies with the community**Happy validating!** 🏥✨